<a href="https://colab.research.google.com/github/Abishek18032007/Assignment1-AIML-/blob/main/area.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
path = kagglehub.dataset_download("debashis74017/stock-market-data-nifty-50-stocks-1-min-data")

100%|██████████| 1.02G/1.02G [00:10<00:00, 108MB/s]

Extracting files...


In [ ]:
import os
import pandas as pd

# The 'path' variable from the previous cell holds the path to the downloaded dataset.
# We need to recursively search for CSV files within this path.

df = None
csv_files = []

# Recursively search for .csv files starting from the 'path' directory
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            csv_files.append(os.path.join(root, file))

if csv_files:
    # Assuming the first CSV found is the main data file, or you can add logic to select a specific one
    data_file_path = csv_files[0]
    print(f"Found and loading '{data_file_path}' into a DataFrame.")
    df = pd.read_csv(data_file_path)
    print("DataFrame loaded successfully.")
else:
    print(f"No CSV files found in the directory '{path}' or its subdirectories.")
    print(f"Contents of the downloaded directory '{path}':")
    for item in os.listdir(path):
        print(f"  - {item}")
    df = None # Ensure df is None if no CSV is found

Found and loading '/root/.cache/kagglehub/datasets/debashis74017/stock-market-data-nifty-50-stocks-1-min-data/versions/14/TITAN_minute.csv' into a DataFrame.
DataFrame loaded successfully.


In [ ]:
if df is not None:
    print("\nFirst 5 rows of the DataFrame:")
    display(df.head())
    print("\nDataFrame Info:")
    display(df.info())
else:
    print("DataFrame was not loaded because no CSV file was found.")


First 5 rows of the DataFrame:


,date,open,high,low,close,volume
0,2015-02-02 09:15:00,428.00,429.1,428.00,428.70,972
1,2015-02-02 09:16:00,428.85,429.5,428.70,428.70,4494
2,2015-02-02 09:17:00,428.70,429.7,428.60,429.50,3332
3,2015-02-02 09:18:00,429.15,429.9,429.15,429.85,1273
4,2015-02-02 09:19:00,429.25,429.7,429.25,429.65,304



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015192 entries, 0 to 1015191
Data columns (total 6 columns):
 #   Column  Non-Null Count    Dtype  
---  ------  --------------    -----  
 0   date    1015192 non-null  object 
 1   open    1015192 non-null  float64
 2   high    1015192 non-null  float64
 3   low     1015192 non-null  float64
 4   close   1015192 non-null  float64
 5   volume  1015192 non-null  int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 46.5+ MB


None

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, confusion_matrix
import numpy as np

# Ensure 'date' is a datetime object
df['date'] = pd.to_datetime(df['date'])

# Sort by date to ensure correct time-series operations
df = df.sort_values(by='date').reset_index(drop=True)

# --- Feature Engineering and Target Creation ---

# Target for Regression: Predict the next minute's close price
df['next_close'] = df['close'].shift(-1)

# Target for Classification: Predict if the price will go up (1) or down/same (0)
df['price_direction'] = (df['next_close'] > df['close']).astype(int)

# Drop the last row as it will have NaN for 'next_close' and 'price_direction'
df.dropna(inplace=True)

# Extract time-based features
df['hour'] = df['date'].dt.hour
df['minute'] = df['date'].dt.minute
df['dayofweek'] = df['date'].dt.dayofweek

# Additional features (examples)
df['high_low_diff'] = df['high'] - df['low']
df['open_close_diff'] = df['close'] - df['open']

print("DataFrame after feature engineering and target creation:")
display(df.head())
display(df.info())


DataFrame after feature engineering and target creation:


,date,open,high,low,close,volume,next_close,price_direction,hour,minute,dayofweek,high_low_diff,open_close_diff
0,2015-02-02 09:15:00,428.00,429.1,428.00,428.70,972,428.70,0,9,15,0,1.10,0.70
1,2015-02-02 09:16:00,428.85,429.5,428.70,428.70,4494,429.50,1,9,16,0,0.80,-0.15
2,2015-02-02 09:17:00,428.70,429.7,428.60,429.50,3332,429.85,1,9,17,0,1.10,0.80
3,2015-02-02 09:18:00,429.15,429.9,429.15,429.85,1273,429.65,0,9,18,0,0.75,0.70
4,2015-02-02 09:19:00,429.25,429.7,429.25,429.65,304,430.00,1,9,19,0,0.45,0.40


<class 'pandas.core.frame.DataFrame'>
Index: 1015191 entries, 0 to 1015190
Data columns (total 13 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   date             1015191 non-null  datetime64[ns]
 1   open             1015191 non-null  float64       
 2   high             1015191 non-null  float64       
 3   low              1015191 non-null  float64       
 4   close            1015191 non-null  float64       
 5   volume           1015191 non-null  int64         
 6   next_close       1015191 non-null  float64       
 7   price_direction  1015191 non-null  int64         
 8   hour             1015191 non-null  int32         
 9   minute           1015191 non-null  int32         
 10  dayofweek        1015191 non-null  int32         
 11  high_low_diff    1015191 non-null  float64       
 12  open_close_diff  1015191 non-null  float64       
dtypes: datetime64[ns](1), float64(7), int32(3), int64(2)
memory us

None

In [ ]:
# --- Prepare Data for Regression Task ---

X_reg = df[['open', 'high', 'low', 'close', 'volume', 'hour', 'minute', 'dayofweek', 'high_low_diff', 'open_close_diff']]
y_reg = df['next_close']

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42, shuffle=False) # shuffle=False for time-series data

print("Regression Data Split Shapes:")
print(f"X_reg_train: {X_reg_train.shape}, y_reg_train: {y_reg_train.shape}")
print(f"X_reg_test: {X_reg_test.shape}, y_reg_test: {y_reg_test.shape}")

# --- Prepare Data for Classification Task ---

X_cls = df[['open', 'high', 'low', 'close', 'volume', 'hour', 'minute', 'dayofweek', 'high_low_diff', 'open_close_diff']]
y_cls = df['price_direction']

X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, shuffle=False) # shuffle=False for time-series data

print("\nClassification Data Split Shapes:")
print(f"X_cls_train: {X_cls_train.shape}, y_cls_train: {y_cls_train.shape}")
print(f"X_cls_test: {X_cls_test.shape}, y_cls_test: {y_cls_test.shape}")


Regression Data Split Shapes:
X_reg_train: (812152, 10), y_reg_train: (812152,)
X_reg_test: (203039, 10), y_reg_test: (203039,)

Classification Data Split Shapes:
X_cls_train: (812152, 10), y_cls_train: (812152,)
X_cls_test: (203039, 10), y_cls_test: (203039,)


### 1. Linear Regression (for predicting `next_close`)

In [ ]:
print("Training Linear Regression Model...")
lin_reg_model = LinearRegression()
lin_reg_model.fit(X_reg_train, y_reg_train)

y_reg_pred = lin_reg_model.predict(X_reg_test)

mse = mean_squared_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mse)

print(f"Linear Regression - Mean Squared Error: {mse:.4f}")
print(f"Linear Regression - Root Mean Squared Error: {rmse:.4f}")


Training Linear Regression Model...
Linear Regression - Mean Squared Error: 7.0493
Linear Regression - Root Mean Squared Error: 2.6550


### 2. Logistic Regression (for predicting `price_direction`)

In [ ]:
print("Training Logistic Regression Model...")
log_reg_model = LogisticRegression(solver='liblinear', random_state=42)
log_reg_model.fit(X_cls_train, y_cls_train)

y_cls_pred_log_reg = log_reg_model.predict(X_cls_test)

print("Logistic Regression - Accuracy:", accuracy_score(y_cls_test, y_cls_pred_log_reg))
print("Logistic Regression - Classification Report:\n", classification_report(y_cls_test, y_cls_pred_log_reg))
print("Logistic Regression - Confusion Matrix:\n", confusion_matrix(y_cls_test, y_cls_pred_log_reg))


Training Logistic Regression Model...
Logistic Regression - Accuracy: 0.557907594107536
Logistic Regression - Classification Report:
               precision    recall  f1-score   support

           0       0.57      0.62      0.59    106069
           1       0.54      0.49      0.52     96970

    accuracy                           0.56    203039
   macro avg       0.56      0.56      0.55    203039
weighted avg       0.56      0.56      0.56    203039

Logistic Regression - Confusion Matrix:
 [[65420 40649]
 [49113 47857]]


### 3. Decision Tree Classifier (for predicting `price_direction`)

In [ ]:
print("Training Decision Tree Classifier Model...")
dtree_model = DecisionTreeClassifier(random_state=42)
dtree_model.fit(X_cls_train, y_cls_train)

y_cls_pred_dtree = dtree_model.predict(X_cls_test)

print("Decision Tree - Accuracy:", accuracy_score(y_cls_test, y_cls_pred_dtree))
print("Decision Tree - Classification Report:\n", classification_report(y_cls_test, y_cls_pred_dtree))
print("Decision Tree - Confusion Matrix:\n", confusion_matrix(y_cls_test, y_cls_pred_dtree))


Training Decision Tree Classifier Model...
Decision Tree - Accuracy: 0.5104044050650368
Decision Tree - Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.59      0.56    106069
           1       0.49      0.42      0.45     96970

    accuracy                           0.51    203039
   macro avg       0.51      0.51      0.50    203039
weighted avg       0.51      0.51      0.51    203039

Decision Tree - Confusion Matrix:
 [[62838 43231]
 [56176 40794]]


### 4. Random Forest Classifier (for predicting `price_direction`)

In [ ]:
print("Training Random Forest Classifier Model...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_cls_train, y_cls_train)

y_cls_pred_rf = rf_model.predict(X_cls_test)

print("Random Forest - Accuracy:", accuracy_score(y_cls_test, y_cls_pred_rf))
print("Random Forest - Classification Report:\n", classification_report(y_cls_test, y_cls_pred_rf))
print("Random Forest - Confusion Matrix:\n", confusion_matrix(y_cls_test, y_cls_pred_rf))


Training Random Forest Classifier Model...
Random Forest - Accuracy: 0.5260368697639369
Random Forest - Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.87      0.66    106069
           1       0.51      0.15      0.23     96970

    accuracy                           0.53    203039
   macro avg       0.52      0.51      0.44    203039
weighted avg       0.52      0.53      0.45    203039

Random Forest - Confusion Matrix:
 [[92398 13671]
 [82562 14408]]
